# 종목별 주주총회 일정과 안건

종목과 권리기준일 범위를 입력하면 SEIBRO Open API에서 주총 기준일, 개최 정보,
안건을 조회합니다. 기준일과 실제 개최일을 각각 표시합니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/seibro-api/blob/main/scenarios/1_stock/02_shareholder_meetings.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `SEIBRO_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 첫 셀은 패키지를 설치하고, 두 번째 셀은 모든 시나리오에 공통인 준비 코드입니다(접어 두어도 됩니다). SEIBRO Open API 키는 [SEIBRO OpenPlatform](https://seibro.or.kr/OpenPlatform/)에서 신청합니다.

In [1]:
# Colab에서만: 패키지 설치 (로컬은 이 레포를 pip install -e . 로 설치해 두면 됨)
import sys
if "google.colab" in sys.modules:
    !pip -q install "git+https://github.com/beaten-by-the-market/seibro-api.git" koreanize-matplotlib

In [2]:
SECRETS = ['SEIBRO_API_KEY']
# ── SEIBRO 공통 셀 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 아래 SECRETS 이름으로 값을 넣고 "노트북 액세스"를 켜세요.
#        SEIBRO_API_KEY = 예탁결제원 SEIBRO Open API 인증키 (https://seibro.or.kr/OpenPlatform)
#        DART_API_KEY = OpenDART 인증키 (DART 데이터를 쓰는 노트북만)
#        KRX_ID / KRX_PW = KRX 정보데이터시스템 계정 (KRX 데이터를 쓰는 노트북만)
# 로컬: 환경변수 또는 작업 폴더(레포 루트)의 .env 를 읽습니다.
# 키·비밀번호 값은 어디에도 출력하지 않습니다. 있는지 여부만 보여 줍니다.
import os, sys, warnings
from datetime import date, datetime, timedelta

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

if IN_COLAB:
    from google.colab import userdata
    for _name in SECRETS:  # noqa: F821  (빌더가 이 셀 맨 위에 SECRETS 목록을 넣음)
        if os.environ.get(_name):
            continue
        try:
            _v = userdata.get(_name)
        except Exception:
            _v = None
        if not _v:
            from getpass import getpass
            _v = getpass(f"{_name}: ")
        os.environ[_name] = _v.strip()
else:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))

_missing = [
    n for n in SECRETS  # noqa: F821
    if not os.environ.get(n) and (IN_COLAB or not n.startswith("KRX_"))
]
if _missing:
    raise RuntimeError(f"{_missing} 가 없습니다. Colab Secrets 또는 .env 에 넣어 주세요.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def won(v, digits=1):
    """원 → '1.2조' / '5,027억' / '3,400만'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


def shares(v):
    """주식수 → '23,607,980주'."""
    return "-" if v is None or pd.isna(v) else f"{int(v):,}주"


# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

def _status(n):
    if os.environ.get(n):
        return "있음"
    # 로컬에서 krx-data-api를 편집 설치(pip install -e)했다면 그 레포의 .env를 패키지가 직접 읽는다
    return "없음" if IN_COLAB or not n.startswith("KRX_") else "krx-data-api 자체 .env 사용"


print("SEIBRO 준비 완료", "(Colab)" if IN_COLAB else "(로컬)",
      "| " + ", ".join(f"{n}: {_status(n)}" for n in SECRETS))  # noqa: F821

SEIBRO 준비 완료 (로컬) | SEIBRO_API_KEY: 있음


In [3]:
from IPython.display import display

from seibro_api import get_shareholder_meetings

STOCK_CODE = "005930"  # 삼성전자
START_DT = "20250101"
END_DT = "20261231"

In [4]:
result = get_shareholder_meetings(STOCK_CODE, START_DT, END_DT)

for name, labels in (
    ("record_dates", {
        "ISSUCO_CUSTNO": "발행회사고객번호", "REP_SECN_NM": "회사명",
        "RGT_STD_DT": "권리기준일", "RGT_RACD_NM": "일정사유",
    }),
    ("meetings", {
        "ISSUIN_NO": "발행회차", "RGT_STD_DT": "권리기준일", "GMET_DT": "개최일",
        "GMET_TTM": "개최시각", "GMET_PLACE": "장소", "EVOTE_YN": "전자투표",
    }),
    ("agendas", {
        "ISSUIN_NO": "발행회차", "RGT_STD_DT": "권리기준일", "GMET_DT": "개최일",
        "AGENDA_NO": "안건번호", "AGENDA_NM": "안건명", "DTAIL_AGENDA_NO": "세부안건번호",
        "DTAIL_AGENDA_NM": "세부안건명", "GMET_RESULT_TPCD": "결과코드(원문)",
    }),
):
    print(f"{name} · {len(result[name])}건")
    if not result[name].empty:
        display(result[name].filter(items=labels).rename(columns=labels))

record_dates · 1건


,발행회사고객번호,회사명,권리기준일,일정사유
0,593,삼성전자,20251231,정기총회


meetings · 1건


,발행회차,권리기준일,개최일,개최시각,장소,전자투표
0,00593,20251231,20260318,0900,"경기도 수원시 영통구 광교중앙로 140 수원컨벤션센터 3층 컨벤션홀, 1층 전시홀",Y


agendas · 9건


,발행회차,권리기준일,개최일,안건번호,안건명,세부안건번호,세부안건명,결과코드(원문)
0,00593,20251231,20260318,002,정관변경,,집중투표제 배제 조항 삭제,1
1,00593,20251231,20260318,002,정관변경,,개정 상법 반영,1
2,00593,20251231,20260318,002,정관변경,,이사의 임기 조문 정비,1
3,00593,20251231,20260318,002,정관변경,,주식소각 조문 정비,1
4,00593,20251231,20260318,001,재무제표승인,,,1
5,00593,20251231,20260318,500,사내이사 선임,,김용관,1
6,00593,20251231,20260318,222,감사위원인 사외이사 선임,,허은녕,1
7,00593,20251231,20260318,301,이사 보수한도액 승인,,,1
8,00593,20251231,20260318,0,자기주식 보유·처분 계획 승인의 건,,,1


- 조회 기간은 **권리기준일** 기준입니다. 개최일이 조회 기간 밖이어도 해당 기준일의 주총이면 표시합니다.
- 기준일이 같아도 주총이 여러 건이면 발행회차를 함께 확인합니다.
- 소집공고의 정관 신구조문은 DART 원문에서 확인할 수 있습니다.